In [ ]:
import pandas as pd
import sqlite3
import numpy as np
import re

db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)

In [ ]:
# HELPER FUNCTIONS
# ==============================================================================
def clean_fda_dates(date_series):
    date_str = date_series.astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', np.nan)
    date_str = date_str.apply(
        lambda x: x + '0101' if pd.notnull(x) and len(str(x)) == 4 else 
                 (x + '01' if pd.notnull(x) and len(str(x)) == 6 else x)
    )
    return pd.to_datetime(date_str, format='%Y%m%d', errors='coerce')

def clean_text_column(text_series):
    return text_series.astype(str).str.lower().str.strip().replace('nan', np.nan)

def professional_drug_clean(text_series):
    """UPGRADE: Removes dosages (mg, ml, g) from drug names using Regex."""
    series = clean_text_column(text_series)
    # Regex to remove patterns like "500 mg", "10ml", "1.5 g"
    dosage_pattern = r'\b\d+(\.\d+)?\s*(mg|g|mcg|ml|iu|u|meq|%)\b'
    series = series.str.replace(dosage_pattern, '', regex=True, flags=re.IGNORECASE)
    # Remove extra spaces left behind
    return series.str.replace(r'\s+', ' ', regex=True).str.strip()

In [ ]:
# TABLE 1: DEMO (Patient Demographics)

df_demo = pd.read_sql_query("SELECT * FROM demo", conn)

# UPGRADE 1: Complex De-duplication (Keep only the latest follow-up per patient)
df_demo = df_demo.sort_values('primaryid', ascending=True)
df_demo = df_demo.drop_duplicates(subset=['caseid'], keep='last')

# 1. Drop useless administrative columns
columns_to_drop = ['to_mfr', 'lit_ref', 'auth_num']
df_demo.drop(columns=[col for col in columns_to_drop if col in df_demo.columns], inplace=True)

# 2. Weight Normalization (Strictly to KG)
df_demo['wt'] = pd.to_numeric(df_demo['wt'], errors='coerce')
df_demo.loc[df_demo['wt_cod'] == 'LBS', 'wt'] /= 2.20462
df_demo.loc[df_demo['wt_cod'] == 'GMS', 'wt'] /= 1000.0
df_demo['wt_cod'] = 'KG'
df_demo.loc[(df_demo['wt'] > 300) | (df_demo['wt'] < 1), 'wt'] = np.nan

# 3. Age Normalization (Strictly to Years)
df_demo['age'] = pd.to_numeric(df_demo['age'], errors='coerce')
conditions = [
    df_demo['age_cod'] == 'DEC', df_demo['age_cod'] == 'MON',
    df_demo['age_cod'] == 'WK', df_demo['age_cod'] == 'DY', df_demo['age_cod'] == 'HR'
]
choices = [
    df_demo['age'] * 10, df_demo['age'] / 12.0,
    df_demo['age'] / 52.1429, df_demo['age'] / 365.25, df_demo['age'] / 8766.0
]
df_demo['age'] = np.select(conditions, choices, default=df_demo['age'])
df_demo['age_cod'] = 'YR'
df_demo.loc[(df_demo['age'] > 120) | (df_demo['age'] < 0), 'age'] = np.nan

# 4. Categorical & Date Cleaning
df_demo['sex'] = df_demo['sex'].replace({'UNK': np.nan}).fillna('UNK')
date_columns = ['event_dt', 'init_fda_dt', 'fda_dt', 'mfr_dt', 'rept_dt']
for col in date_columns:
    if col in df_demo.columns:
        df_demo[col] = clean_fda_dates(df_demo[col])

df_demo.to_sql('demo_clean', conn, if_exists='replace', index=False)
print(f" DEMO table cleaned: {len(df_demo)} records saved.")
del df_demo

In [ ]:
# TABLE 2: DRUG (Medications & Doses)
# ==============================================================================
print("\n--- Processing DRUG Table ---")
df_drug = pd.read_sql_query("SELECT * FROM drug", conn)

# UPGRADE 2: Apply Professional Regex text cleaning
df_drug['drugname'] = professional_drug_clean(df_drug['drugname'])
if 'prod_ai' in df_drug.columns:
    df_drug['prod_ai'] = professional_drug_clean(df_drug['prod_ai'])

# Consolidate Drug Name
if 'prod_ai' in df_drug.columns:
    df_drug['final_drug_name'] = df_drug['prod_ai'].fillna(df_drug['drugname'])
else:
    df_drug['final_drug_name'] = df_drug['drugname']

# Clean Categorical Indicators
for col in ['dechal', 'rechal', 'role_cod']:
    if col in df_drug.columns:
        df_drug[col] = df_drug[col].astype(str).str.upper().replace('NAN', 'UNK')

df_drug.to_sql('drug_clean', conn, if_exists='replace', index=False)
print(f" DRUG table cleaned: {len(df_drug)} records saved.")
del df_drug

In [ ]:
# TABLE 3: THER (Therapy Dates & Duration)
# ==============================================================================
print("\n--- Processing THER Table ---")
df_ther= pd.read_sql_query("SELECT * FROM ther", conn)

df_ther['start_dt'] = clean_fda_dates(df_ther['start_dt'])
df_ther['end_dt'] = clean_fda_dates(df_ther['end_dt'])
df_ther['dur'] = pd.to_numeric(df_ther['dur'], errors='coerce')

conditions = [
    df_ther['dur_cod'] == 'YR', df_ther['dur_cod'] == 'MON',
    df_ther['dur_cod'] == 'WK', df_ther['dur_cod'] == 'HR',
    df_ther['dur_cod'] == 'MIN', df_ther['dur_cod'] == 'SEC'
]
choices = [
    df_ther['dur'] * 365.25, df_ther['dur'] * 30.4368,
    df_ther['dur'] * 7.0, df_ther['dur'] / 24.0,
    df_ther['dur'] / 1440.0, df_ther['dur'] / 86400.0
]
df_ther['dur'] = np.select(conditions, choices, default=df_ther['dur'])
df_ther['dur_cod'] = 'DAY'
df_ther.loc[(df_ther['dur'] < 0) | (df_ther['dur'] > 36500), 'dur'] = np.nan

calculated_duration = (df_ther['end_dt'] - df_ther['start_dt']).dt.days
mask_missing_dur = (
    df_ther['dur'].isna() & df_ther['start_dt'].notna() & 
    df_ther['end_dt'].notna() & (df_ther['end_dt'] >= df_ther['start_dt'])
)
df_ther.loc[mask_missing_dur, 'dur'] = calculated_duration[mask_missing_dur]

df_ther.to_sql('ther_clean', conn, if_exists='replace', index=False)
print(f" THER table cleaned: {len(df_ther)} records saved.")
del df_ther

In [ ]:
# REMAINING TABLES (REAC, OUTC, INDI, RPSR)
# ==============================================================================
tables_to_clean = {
    'REAC': ('reac', 'reac_clean', ['pt', 'drug_rec_act']),
    'OUTC': ('outc', 'outc_clean', []),
    'INDI': ('indi', 'indi_clean', ['indi_pt']),
    'RPSR': ('rpsr', 'rpsr_clean', [])
}

for table_name, (raw_table, clean_table, text_cols) in tables_to_clean.items():
    print(f"\n--- Processing {table_name} Table ---")
    df = pd.read_sql_query(f"SELECT * FROM {raw_table}", conn)
    
    # Clean standard text columns
    for col in text_cols:
        if col in df.columns:
            df[col] = clean_text_column(df[col])
            
    # Table-specific standardizations
    if table_name == 'OUTC' and 'outc_cod' in df.columns:
        df['outc_cod'] = df['outc_cod'].astype(str).str.upper().str.strip().replace('NAN', np.nan)
        df.dropna(subset=['outc_cod'], inplace=True)
        
    if table_name == 'RPSR' and 'rpsr_cod' in df.columns:
        df['rpsr_cod'] = df['rpsr_cod'].astype(str).str.upper().str.strip().replace('NAN', np.nan)

    df.to_sql(clean_table, conn, if_exists='replace', index=False)
    print(f"✅ {table_name} table cleaned: {len(df)} records saved.")
    del df

conn.close()